# Language Table, Layer 2: your evaluation runs versus the paper

Layer 1 (`L1_tables.ipynb`) recomputes the paper tables from the released per-episode results. This notebook does the same for the evaluation runs you make with `L2_eval.sh` and prints them next to the paper and the released results:

| # | Run | Command | Output read here |
|---:|---|---|---|
| 1 | WMM, plain policy | `apps/langtable/L2_eval.sh plain 0` | `L2_outputs/plain/<family>/<NNNN>/meta.json` |
| 2 | WMM, MPC | `apps/langtable/L2_eval.sh mpc 0,1` | `L2_outputs/mpc/<family>/ep_<N>/meta.json` |
| 3 | LangTable orig. (public checkpoint of Lynch et al. 2022, its own `eval/main.py`) | `apps/langtable/L2_eval.sh orig` | `L2_outputs/orig/videos/<family>_<N>_<success\|failure>.mp4` |

To compare another run, for example a checkpoint you trained in Layer 3 and evaluated with `RUN_NAME=plain_l3`, change `RUNS` in the next cell.

The table code is shared with Layer 1 (`lt_tables.py`). A run that has not finished all 250 episodes is reported as *not complete* and its cells are left out.

In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from lt_tables import (COLUMNS, DIGITS, FAMILIES, PAPER_TABLE1, PAPER_TABLE2, STATS, check_results_json,
                       find_repo_root, langtable_data, load_episodes, load_orig_episodes, policy_stats,
                       success_rates)

# Folders under L2_outputs/ to compare (the RUN_NAME of L2_eval.sh; default: the row name).
RUNS = {'plain': 'plain', 'mpc': 'mpc', 'orig': 'orig'}

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA = langtable_data(REPO_ROOT)
L1, L2 = DATA / 'L1_results', DATA / 'L2_outputs'
if not L1.is_dir():
    raise FileNotFoundError(f'{L1} not found (see apps/langtable/README.md)')


def wmm_run_complete(run_dir):
    """results.json lists all five families with 50 episodes each (the plain script rewrites it after every family)."""
    f = run_dir / 'results.json'
    return f.is_file() and all(json.loads(f.read_text())['rewards'].get(folder, {}).get('total') == 50
                               for folder in FAMILIES.values())


released, yours = {}, {}
for mode in ['plain', 'mpc']:
    released[mode] = load_episodes(L1 / mode, mode)
    run_dir = L2 / RUNS[mode]
    if wmm_run_complete(run_dir):
        yours[mode] = load_episodes(run_dir, mode)
        check_results_json(yours[mode], run_dir, mode)
    print(f"{mode:5s} (L2_outputs/{RUNS[mode]}):", 'complete (250 episodes)' if mode in yours else 'not complete')
orig_dir = L2 / RUNS['orig']
orig = load_orig_episodes(orig_dir) if len(list((orig_dir / 'videos').glob('*.mp4'))) == 5 * 50 else None
print(f"orig  (L2_outputs/{RUNS['orig']}):", 'complete (250 episodes)' if orig is not None else 'not complete')

## 1. Same scenes, different rollouts



In [ ]:
rows = []
for mode, run in yours.items():
    a = released[mode].set_index(['family', 'ep'])
    b = run.set_index(['family', 'ep'])
    same_instr = a.instruction == b.instruction
    is_sep = a.index.get_level_values('family') == 'Sep'
    differ = [f'{fam} ep {ep}' for (fam, ep), same in same_instr[~is_sep].items() if not same]
    if mode == 'plain':
        assert not differ, f'plain: instructions differ from the released run outside the separate family: {differ}'
    rows.append({'run': mode, 'episodes': len(a),
                 'same instruction (B2B..B2RL)': f'{int(same_instr[~is_sep].sum())}/{int((~is_sep).sum())}',
                 'same wording (Sep)': f'{int(same_instr[is_sep].sum())}/{int(is_sep.sum())}',
                 'same success': int((a.success == b.success).sum()),
                 'same success and steps': int(((a.success == b.success) & (a.steps == b.steps)).sum()),
                 'other instruction (B2B..B2RL)': ', '.join(differ) or '-'})
print(pd.DataFrame(rows).to_string(index=False) if rows else 'no complete WMM run yet')

## 2. Table 1: success rate per family

For *LangTable orig.* the paper value is quoted from the LCB paper.

In [ ]:
def stderr(p, n):
    return 100.0 * math.sqrt((p / 100.0) * (1 - p / 100.0) / n)


table1_runs = []
if 'plain' in yours:
    table1_runs.append(('WMM (ours)', success_rates(released['plain']), success_rates(yours['plain'])))
if orig is not None:
    table1_runs.append(('LangTable orig.', None, success_rates(orig)))
rows = []
for method, rel, mine in table1_runs:
    for i, col in enumerate(COLUMNS):
        p, v = PAPER_TABLE1[method][i], mine[col]
        rows.append({'row': method, 'column': col, 'paper': p,
                     'released results': None if rel is None else round(rel[col], 1),
                     'your run': round(v, 1), 'your run - paper': round(v - p, 1),
                     'stderr': round(stderr(v, 250 if col == 'Avg' else 50), 1)})
tab1 = pd.DataFrame(rows)
print(tab1.to_string(index=False) if len(tab1) else 'no Table 1 run yet')

## 3. Table 2: plain policy versus MPC

Same statistics as Layer 1: success rate, and mean / median / min / max steps over successful episodes only.

In [ ]:
rows = []
for (fam, variant), paper_vals in PAPER_TABLE2.items():
    mode = variant.lower()
    if mode not in yours:
        continue
    rel = policy_stats(released[mode][released[mode].family == fam])
    mine = policy_stats(yours[mode][yours[mode].family == fam])
    for stat, p, a, b in zip(STATS, paper_vals, rel, mine):
        d = DIGITS[stat]
        rows.append({'row': f'{fam} {variant}', 'stat': stat, 'paper': p, 'released results': round(a, d),
                     'your run': round(b, d), 'your run - paper': round(b - p, d)})
tab2 = pd.DataFrame(rows)
print(tab2.to_string(index=False) if len(tab2) else 'no Table 2 run yet')
if 'plain' in yours and 'mpc' in yours:
    for fam in ['B2B', 'B2AL']:
        get = lambda v, s: tab2[(tab2.row == f'{fam} {v}') & (tab2.stat == s)]['your run'].item()
        print(f'{fam}: MPC - Plain success {get("MPC", "Succ.") - get("Plain", "Succ."):+.1f} points, '
              f'mean steps {get("MPC", "Mean steps") - get("Plain", "Mean steps"):+.1f} (paper: MPC higher success, fewer steps)')

In [ ]:
if len(tab1):
    methods = list(dict.fromkeys(tab1.row))
    fig, axes = plt.subplots(1, len(methods), figsize=(6 * len(methods), 3.5), squeeze=False)
    width = 0.27
    for ax, method in zip(axes[0], methods):
        sub = tab1[tab1.row == method]
        x = list(range(len(sub)))
        ax.bar([i - width for i in x], sub.paper, width, label='paper')
        if sub['released results'].notna().all():
            ax.bar(x, sub['released results'], width, label='released results')
        ax.bar([i + width for i in x], sub['your run'], width, yerr=sub.stderr, capsize=3, label='your run')
        ax.set_xticks(x, sub.column)
        ax.set_ylim(0, 105)
        ax.set_ylabel('success rate (%)')
        ax.set_title(method)
        ax.legend(loc='lower left', fontsize=8)
    fig.tight_layout()
    plt.show()